---
title: "Concepts in Cryptography"
author: "Vahram Poghosyan"
date: "2025-11-05"
categories: ["Cryptography"]
format:
  html:
    code-fold: true
jupyter: python3
include-after-body:
  text: |
    <script type="application/javascript" src="../../javascript/light-dark.js"></script>
---

# Hash

A hash function is a mathematical function that takes an input (or 'message') and returns a fixed-size string of bytes. The output, typically a 'digest', appears random. Hash functions are designed to be fast to compute, but their inverse is difficult to compute (so a hashed input cannot be easily reverse-engineered).

Examples of hash functions include SHA-256 (Secure Hash Algorithm), SHA-1, MD5 (which is insecure by modern standards), and Argon2.

Hashing allows us to:

- **Verify data integrity**: If even a single bit of the input changes, the hash output will change significantly.
- **Store passwords securely**: Although hashing itself is not enough for password security (see [below](#salt)), it is a step that's involved in securing passwords. Instead of storing the actual password, systems store the hash of the password (or, rather, the hash of the salt + password). When a user logs in, the system hashes the entered password and compares it to the stored hash.

Example implementation in Node.js using the `crypto` module:

```javascript
const crypto = require('crypto');

function hash(input) {
    const hash = crypto.createHash('sha256');
    const digest = hash.update(input).digest('hex'); // Another option is base64
    return digest;
}
```
Example usage:

```javascript
const password = "helloworld!"
const hash1 = hash(password);
console.log(hash1); // Outputs the SHA-256 hash of "helloworld!"

const password2 = "helloworld?"
const hash2 = hash(password2);
console.log(hash2); // Outputs a different SHA-256 hash

const match = hash1 === hash2;
console.log(match); // Returns false
```

# Salt

A hash function, by itself, is vulnerable to certain attacks. So, it's not sufficient to store hashed passwords. Hackers can go to a **rainbow table** that has a bunch of pre-computed hashes for commonly used passwords and identify the commonly used passwords in our system (leaving the technologically-impaired users of our service at risk). To mitigate this, we can add a 'salt' to the input before hashing. A salt is a random value that is unique for each input. By adding a salt, even if two users have the same password, their hashes will be different (so no rainbow table will contain this value). This makes it almost impossible for attackers to use pre-computed tables to crack passwords.

The intentional slowness of `scryptSync` makes it computationally expensive for attackers to try millions of password combinations, while `createHash` is optimized for speed when verifying file integrity or creating digital signatures. So, we use `scryptSync` for password hashing and `createHash` for data integrity checks.

Example implementation in Node.js using the `crypto` module with salt:

```javascript
const {crypto, scryptSync, randomBytes } = require('crypto');

function signup(email, password) {
    const salt = randomBytes(16).toString('hex'); // Generate a random salt
    const hash = scryptSync(password, salt, 64).toString('hex'); // Hash the password with the salt
    const storedPassword = `${salt}:${hash}`; // Store the salt and hash together
    // Persistence Layer: Save email and storedPassword to database
}

function login(email, password) {
    // Persistence Layer: Retrieve storedPassword from database using email...
    const [salt, hash] = storedPassword.split(':'); // Split the stored password into salt and hash
    const hashVerify = scryptSync(password, salt, 64).toString('hex'); // Hash the entered password with the retrieved salt
    if (hash === hashVerify) {
        console.log('Login successful');
    } else {
        console.log('Login failed');
    }
}
```
To further prevent timing attacks, it's advisable to use constant-time comparison functions when comparing hashes. This ensures that the time taken to compare two hashes does not depend on how many characters match, making it harder for attackers to infer information based on timing. For this we use the `crypto.timingSafeEqual` method:

```javascript
function constantTimeCompare(hash1, hash2) {
    return crypto.timingSafeEqual(Buffer.from(hash1, 'hex'), Buffer.from(hash2, 'hex'));
}
```
The method expects `Buffer` inputs, so we convert the hexadecimal strings to `Buffer` objects before comparison. What is a Buffer? A `Buffer` is a built-in class in Node.js that provides a way to work with binary data directly. The method expects binary data, so we convert the hexadecimal strings to `Buffer` objects before comparison.

```javascript
function login(email, password) {
    // Persistence Layer: Retrieve storedPassword from database using email...
    const [salt, hash] = storedPassword.split(':'); // Split the stored password into salt and hash
    const hashVerify = scryptSync(password, salt, 64).toString('hex'); // Hash the entered password with the retrieved salt
    if (constantTimeCompare(hash, hashVerify)) {
        console.log('Login successful');
    } else {
        console.log('Login failed');
    }
}
```

Even with the added salt, however, hashing alone cannot provide **authenticity** (i.e. the security pillar that says the party we're communicating with is who they say they are). Enter HMAC.

# HMAC (Hash-based Message Authentication Code)

HMAC is a specific construction for creating a message authentication code (MAC) that involves a cryptographic [hash function](#hash) in combination with a secret **cryptographic key** (a MAC key). Without this key, it is computationally infeasible to generate the correct HMAC for a given message. Also, the same message encrypted with a different key produces a wildly different result.

An HMAC can be used to verify both the data integrity and the authenticity of a message.  Meaning, not only does it obscure the message and guarantee that the data has not been tampered with, it also guarantees the sender is trusted. HMAC is widely used in various security protocols including [TLS/SSL](./ssl_the_internets_trust_protocol.ipynb). Note, SSL/TLS uses asymmetric encryption for the handshake, then symmetric encryption for **confidential** communication. But the handshake also establishes a secret MAC key between the client and the server amd SSL relies on HMAC to establish the remaining pillars of secure communication, **authenticity** and **integrity**. There is no strict division here as both symmetric encryption and HMAC offer some guarantee of integrity and authenticity (and even confidentiality, although with HMAC the original message cannot be recovered). 

JWT (JSON Web Tokens) may also use HMAC for signing tokens to ensure that the token has not been tampered with and is from a trusted source.
JWTs can be signed using a **MAC secret key** (with the HMAC algorithm) or a public/private key pair using RSA or ECDSA.
The header of a JWT typically consists of two parts: the type of the token, which is JWT, and the signing algorithm being used, such as HMAC SHA256 or RSA.

```javascript
const { createHmac } = require('crypto');

const key = "super-secret!"
const message = "Hello, World!"

const hmac = createHmac('sha256', key) 
const digest = hmac.update(message).digest('hex');
console.log(digest) // Prints HMAC SHA256 of message "Hello, World!"
```

# Encryption vs Hashing

The difference between hashing and encryption is: In encryption, confidentiality is the main goal while in hashing verifying integrity and authenticity is the goal. With a hash, you don't care about "un-hashing" to reveal the original message (in fact it's computationally infeasible to do so by design). With encryption, however, there's a corresponding decryption operation... We *care* about the contents of the message.

There are two types of encryption, symmetric and asymmetric. See [this article](./ssl_the_internets_trust_protocol.ipynb.ipynb) for their differences.


# Symmetric Encryption

```javascript
const { createCipheriv, randomBytes, createDecipheriv } = require('crypto');

// Creat cipher
const message = "I like turtles.";
const key = randomBytes(32);
const iv = randomBytes(16); // Inititalization vector (prevents identical cyphertext from same input message)
const cipher = createCypheriv('aes256', key, iv);

// Encrypt the message
const encryptedMessage = cipher.update(message, 'utf8', 'hex') + cipher.final('hex') // cipher.final applies padding (if needed)

// Decrypt the message
const decipher = createDecipheriv('aes256', key, iv);
const decryptedMessage = decipher.update(encryptedMessage, 'utf8', 'hex') + decipher.final('hex');
```

However, there is a big limitation to symmetric encryption. Both the sender and receiver need to share a password. This already poses a big security risk. So, we use a pair of public/private keys. 

# Generate Public/Private Keys

Here's how to generate a private/public key pair using the RSA (Rivest-Shamir-Adleman) system:

```javascript
const {generateKeyPairSync } = require('crypto');

// Generate keys: First parameter is the crypto system (e.g. RSA - Rivest-Shamir-Adleman)
const { privateKey, publicKey } = generateKeyPairSync('rsa', {
    modulusLength: 2048, // Length of the key in bits
    publicKeyEncoding: {
        type: 'spki', // Good choice for Node.js (per Node.js docs)
    },
    privateKeyEncoding: {
        type: 'pkcs8', // Goood choice for Node.js, once again
        format: 'pem' // Format of keys, stands for Privacy Enhanced Mail
        cipher: 'aes-256-cbc',
        passphrase: 'top secret' // Optional, adds security
    },
})

console.log(publicKey); // Gives the generated public key
console.log(privateKey); // Gives the generated private key

// Export out to other modules for use...
module.exports = {
    privateKey, publicKey
}
```

# Asymmetric Encryption

When we visit a trusted website over HTTPS, the browser automatically finds the public of this website that's listed in the website's SSL certificate (signed by a CA) within the browser's certificate store. It then uses the CA's public key (also within its certificate store) to verify the SSL signature. Once the signature of the SSL certificate is verified, the server's identity is tied to its public key (the one in the SSL certificate), establishing authenticity. The public key of the server is then used to encrypt a *new*, client-generated cryptographic key in-flight to the server (during the asymmetric step of the TLS handshake). The server then uses its corresponding private key to decrypt the client's new key. Now both the server and the client have a shared symmetric key to symmetrically encrypt all messages in-flight. Only the first step of the TLS/SSL handshake is asymmetric.

```javascript
const { publicEncrypt, privateDecrypt } = require('crypto');
const { publicKey, privateKey } = require('./keypair');

const message = 'the enemy is advancing!'

// Sender uses the server's public key to encrypt data (like, during the asymmetric step of the handshake)
const encryptedData = publicEncrypt(
    publickey,
    Buffer.from(message)
)

// Receiver uses its corresponding server private key to decrypt the data
const decryptedData = privateDecrypt(
    privateKey, 
    encrypteData
)
```

# Signing

We mentioned **signing** in the context of the TLS/SSL handshake above. Signing is nothing but the act of using a private key to sign a hash of the original message. The signature guarantees authenticity, while the hash guarantees the message has not been tampered with. The recipient can then use the corresponding private key to validate the authenticity of the message.

```javascript
const { createSign, createVerify } = require('crypto');
const { publicKey, privateKey } = require('./keypair');

const message = 'this message must be signed.'

// Sign (Sender side)
const signer = createSign('rsa-sha256'); // We provide both the crypto-system and the hash function
signer.update(message);
const signature = signer.sign(privateKey, 'hex');

// Verify (Receiver side)
const verified = createVeriy('rsa-sha256');
verifier.update(message);
const isVerified = verified.verify(publicKey);
```